# Point-mass reprofile — T1 straight-line model with the 350 kW taper

The 2026 energy game is a 4 MJ store feeding a 350 kW motor that tapers to zero at
355 km/h (AR-01/AR-02). This notebook turns 2024 telemetry into effective per-circuit
car parameters — one drive-power and one drag constant that absorb engine, density,
CdA and rolling resistance — then swaps in the 2026 power, taper and drag (×0.45) to
produce each circuit's full-throttle time, terminal speed, deployable energy E_dep and
the lap-time delta. No physical aero constant is sourced; everything effective is fitted
from what the car observably did (F-13). Logic lives in `senna.physics.synth.pointmass`;
this notebook is the narrative and the residual plots.

In [ ]:
import argparse
import asyncio
import os

import numpy as np
import pandas as pd

from senna import config
from senna.data.db import database_url, engine_for
from senna.physics.fits import frames as fit_frames
from senna.physics.synth import plots

synth_cfg = config.load("synth")
physics_2026 = config.load("physics_2026")
CALIB = config.get(config.load("data_split"), "calibration_rounds")

# the battery needs Postgres + the FastF1 cache; export DATABASE_URL first
assert os.environ.get("DATABASE_URL"), "export DATABASE_URL (see .env.example) before running"

In [ ]:
from senna.physics.synth import run

# a narrowed battery run keeps the telemetry load small for the walk-through
args = argparse.Namespace(only="aero,pointmass", circuit="monaco_grand_prix",
                          transforms_out=None, physics_2026_out=None)
transforms = asyncio.run(run.run_battery(args))
transforms["circuits"]["monaco_grand_prix"]["t1_powertrain"]

In [ ]:
from senna.physics.synth.pointmass import taper_kw

# the taper is the whole 2026 story: full electric power held to 290 km/h, gone by 355
taper = {"mgu_k_max_kw": config.get(physics_2026, "power_unit.mgu_k_max_kw"),
         "full_power_to_kmh": config.get(physics_2026, "power_unit.deploy_taper.full_power_to_kmh"),
         "zero_power_at_kmh": config.get(physics_2026, "power_unit.deploy_taper.zero_power_at_kmh")}
speeds = np.linspace(0, 380, 200)
fig, ax = plots.plt.subplots(figsize=(6, 4))
ax.plot(speeds, taper_kw(speeds, **taper))
ax.set_xlabel("speed (km/h)"); ax.set_ylabel("MGU-K power (kW)")
ax.set_title("2026 deployment taper — 350 kW to 290 km/h, zero at 355")
plots.save_figure(fig, "point_mass", "deploy_taper")